# 🇧🇩 Bangla Transactional Document KIE: QLoRA Fine-Tuning & Evaluation
### End-to-End Instruction Tuning on Qwen2.5-3B with Verifier-Grounded Evaluation

This notebook trains and evaluates a lightweight, cost-effective LLM (**Qwen2.5-3B-Instruct**) for **Bangla Key Information Extraction (KIE)** across utility bills, bKash/Nagad mobile receipts, and supermarket POS invoices with **strict span grounding**.

## 1. Setup & Environment Installation

In [ ]:
!nvidia-smi
!pip install -q transformers peft trl bitsandbytes accelerate datasets jsonschema rapidfuzz matplotlib

## 2. Load Training Dataset & System Prompt

In [ ]:
import json
import os
from datasets import load_dataset

SYSTEM_PROMPT = """You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.
If a field is missing, omit it or set it to null. Never hallucinate values not present in the OCR text."""

TRAIN_FILE = "data/sft_train_full.jsonl"
if not os.path.exists(TRAIN_FILE):
    print(f"Warning: {TRAIN_FILE} not found. Please upload or clone the repository!")
else:
    dataset = load_dataset("json", data_files=TRAIN_FILE)
    print(f"Loaded dataset with {len(dataset['train'])} examples!")
    print("Sample:", dataset['train'][0]['messages'][1]['content'][:300], "...")

## 3. Load Model in 4-bit NormalFloat (NF4) with LoRA Adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Run SFT Training with TRL SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "output/qwen2.5_3b_bangla_kie"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete! Model adapters saved to:", OUTPUT_DIR)

## 5. Verify & Evaluate Against Grounded Verifier

In [ ]:
# Run evaluation using the project's verifier module
!python eval/evaluate_predictions.py --test-file data/test/synthetic_test_300.json --predictions results/baselines/predictions_sft_round1.json